# Whisper Small — Incremental Medical ASR Training on a New Conversation Dataset

This notebook trains a new medical Whisper Small model on a complete new conversation dataset.

It starts from the existing fine-tuned v1 model and exports a separate v2 model.

- New CSV: `conversations.csv`
- `EXPECTED_RECORDINGS` recordings: `CONV_001.wav` through `CONV_NNN.wav`
- CSV columns: `recording_id`, `transcript`, `classification`
- `classification` is metadata only and is never included in the Whisper transcript label.
- Starting model:
  `/content/drive/MyDrive/medical-whisper/final_whisper-small_medical_full`
- New output model:
  `/content/drive/MyDrive/medical-whisper/final_whisper-small_medical_full_v2`

The run is **fresh by default**. It does not resume an old v2 checkpoint unless `RESUME_TRAINING` is explicitly changed to `True`.
The starting v1 model is never overwritten.


## 1. Check GPU

In [ ]:
import sys
import os
import gc
import re
import json
import shutil
import inspect
import hashlib
import math
import time
import unicodedata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No NVIDIA GPU is attached. "
        "Select Runtime → Change runtime type → GPU and rerun."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        1
    ),
    "GB"
)


## 2. Install packages

In [ ]:
%pip -q install "transformers==4.57.1" "datasets==4.3.0" "accelerate==1.11.0" jiwer soundfile scipy sentencepiece "librosa==0.11.0" tqdm ipywidgets


## 3. Configuration

In [ ]:
# ============================================================
# INCREMENTAL TRAINING CONFIGURATION
# ============================================================

MAX_RECORDINGS = None          # None = all matched recordings
EXPECTED_RECORDINGS = 0       # REQUIRED: number of conversations in this round's dataset
if EXPECTED_RECORDINGS <= 0:
    raise ValueError("Set EXPECTED_RECORDINGS to the number of conversations in your dataset.")
RANDOM_SEED = 42

MAX_AUDIO_SEC = 25.0
MAX_LABEL_TOKENS = 380
MIN_BOUNDARY_SCORE = 0.05

# Existing trained model — starting checkpoint.
EXISTING_MODEL_NAME = "final_whisper-small_medical_full"

# New model output. The existing model is never overwritten.
NEW_MODEL_NAME = "final_whisper-small_medical_full_v2"

# Keep the same training defaults as the current training notebook.
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-5
WARMUP_STEPS = 50
EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 10

RESUME_TRAINING = False
RETRY_FAILED_PREPROCESSING = True

# A fresh v2 training run uses its own output/local directories.
# Set RESUME_TRAINING=True only when intentionally continuing an interrupted v2 run.

print("Expected new recordings:", EXPECTED_RECORDINGS)
print("Training epochs:", NUM_EPOCHS)
print("Training batch size:", TRAIN_BATCH_SIZE)
print("Evaluation every:", EVAL_STEPS, "steps")
print("Checkpoint every:", SAVE_STEPS, "steps")
print("Resume training:", RESUME_TRAINING)
# Audio and CSV IDs must both be CONV_001 ... CONV_NNN.
# This run starts from the original full medical model and exports a separate v2 model.


## 4. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/medical-whisper")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# New conversation dataset
# ------------------------------------------------------------

NEW_DATA_ROOT = DRIVE_ROOT / "conversation_dataset"
NEW_DATA_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_CSV = NEW_DATA_ROOT / "conversations.csv"
DRIVE_AUDIO = NEW_DATA_ROOT / "audio"

# ------------------------------------------------------------
# Existing trained model — starting checkpoint
# ------------------------------------------------------------

EXISTING_MODEL = DRIVE_ROOT / EXISTING_MODEL_NAME

if not DRIVE_CSV.exists():
    raise FileNotFoundError(
        f"Missing new dataset CSV: {DRIVE_CSV}"
    )

if not DRIVE_AUDIO.exists():
    raise FileNotFoundError(
        f"Missing new audio folder: {DRIVE_AUDIO}"
    )

if not EXISTING_MODEL.exists():
    raise FileNotFoundError(
        f"Existing trained model not found: {EXISTING_MODEL}"
    )

print("New CSV:", DRIVE_CSV)
print("New audio folder:", DRIVE_AUDIO)
print("Existing trained model:", EXISTING_MODEL)
print("New WAV files:", len(list(DRIVE_AUDIO.glob("*.wav"))))


## 5. Copy the new conversation audio to stable Colab local storage

Expected audio filenames:

```text
CONV_001.wav
CONV_002.wav
...
CONV_NNN.wav
```


In [ ]:
LOCAL_ROOT = Path("/content/medical-whisper-incremental-conversations-v2")
LOCAL_AUDIO = LOCAL_ROOT / "audio"

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_AUDIO.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    DRIVE_CSV,
    LOCAL_ROOT / "conversations.csv"
)

wav_files = sorted(DRIVE_AUDIO.glob("*.wav"))

if len(wav_files) != EXPECTED_RECORDINGS:
    raise RuntimeError(
        f"Expected {EXPECTED_RECORDINGS} WAV files in {DRIVE_AUDIO}, "
        f"found {len(wav_files)}."
    )

for i, src in enumerate(wav_files, 1):
    dst = LOCAL_AUDIO / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)

    if i % 50 == 0 or i == len(wav_files):
        print(f"Copied/checked {i}/{len(wav_files)}")

print("Local WAV files:", len(list(LOCAL_AUDIO.glob("*.wav"))))


## 6. Match `CONV_###` CSV rows to `CONV_###.wav` audio and create/reuse a deterministic source split

The new CSV must contain exactly `EXPECTED_RECORDINGS` rows with:

- `recording_id`
- `transcript`
- `classification`

`classification` is retained as metadata but is not included in the Whisper transcript label.


In [ ]:
import soundfile as sf

CSV_PATH = LOCAL_ROOT / "conversations.csv"
df = pd.read_csv(CSV_PATH)

required = {"recording_id", "transcript"}
missing = required - set(df.columns)

if missing:
    raise ValueError(f"CSV missing columns: {sorted(missing)}")

if len(df) != EXPECTED_RECORDINGS:
    raise RuntimeError(
        f"Expected exactly {EXPECTED_RECORDINGS} CSV rows, found {len(df)}."
    )


def normalize_id(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if re.fullmatch(r"\d+\.0", value):
        value = value[:-2]
    return value


df["id_norm"] = df["recording_id"].apply(normalize_id)
df["transcript"] = (
    df["transcript"].fillna("").astype(str).str.strip()
)

if df["id_norm"].duplicated().any():
    duplicate_ids = (
        df.loc[df["id_norm"].duplicated(keep=False), "id_norm"]
        .tolist()
    )
    raise RuntimeError(
        "Duplicate conversation IDs found: "
        f"{sorted(set(duplicate_ids))[:20]}"
    )

expected_ids = {
    f"CONV_{i:03d}"
    for i in range(1, EXPECTED_RECORDINGS + 1)
}
csv_ids = set(df["id_norm"].astype(str))

if csv_ids != expected_ids:
    missing_ids = sorted(expected_ids - csv_ids)
    extra_ids = sorted(csv_ids - expected_ids)
    raise RuntimeError(
        "CSV IDs do not exactly match CONV_001 through CONV_NNN. "
        f"Missing: {missing_ids[:20]}; Extra: {extra_ids[:20]}"
    )

audio_map = {p.stem: p for p in LOCAL_AUDIO.glob("*.wav")}

rows = []
missing_audio = []

for _, row in df.iterrows():
    sample_id = row["id_norm"]

    if not row["transcript"]:
        raise RuntimeError(f"{sample_id}: empty transcript.")

    audio_path = audio_map.get(sample_id)

    if audio_path is None:
        missing_audio.append(sample_id)
        continue

    info = sf.info(str(audio_path))

    rows.append({
        "id": sample_id,
        "audio_path": str(audio_path),
        "transcript": row["transcript"],
        "classification": (
            row["classification"]
            if "classification" in row
            else ""
        ),
        "duration_sec": info.frames / info.samplerate,
        "sample_rate": info.samplerate,
    })

pairs = pd.DataFrame(rows)

print("Matched pairs available:", len(pairs))
print("Missing audio:", len(missing_audio))

if missing_audio:
    print("Missing IDs:", missing_audio[:20])

if len(pairs) != EXPECTED_RECORDINGS:
    raise RuntimeError(
        f"Expected exactly {EXPECTED_RECORDINGS} matched recordings, "
        f"found {len(pairs)}."
    )

pairs = pairs.sort_values("id").reset_index(drop=True).copy()

if MAX_RECORDINGS is not None:
    pairs = pairs.head(int(MAX_RECORDINGS)).copy()

if pairs["id"].duplicated().any():
    raise RuntimeError("Duplicate matched recording IDs.")

# ------------------------------------------------------------
# Recording-level deterministic train/validation/test split.
# ------------------------------------------------------------

SPLIT_FILE = NEW_DATA_ROOT / "incremental_recording_splits.csv"
current_ids = set(pairs["id"].astype(str))

reuse_split = False

if SPLIT_FILE.exists():
    split_df = pd.read_csv(SPLIT_FILE)

    if {"id", "split"}.issubset(split_df.columns):
        saved_ids = set(split_df["id"].astype(str))
        if saved_ids == current_ids and len(split_df) == len(pairs):
            reuse_split = True

if reuse_split:
    pairs = (
        pairs.drop(columns=["split"], errors="ignore")
        .merge(
            split_df[["id", "split"]].astype({"id": str}),
            on="id",
            how="left",
            validate="one_to_one",
        )
    )
    print("Reusing existing deterministic source split:", SPLIT_FILE)
else:
    ids = pairs["id"].astype(str).to_numpy()
    rng = np.random.default_rng(RANDOM_SEED)
    shuffled = ids.copy()
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_train = max(1, int(round(0.80 * n)))
    n_val = max(1, int(round(0.10 * n)))

    if n_train + n_val >= n:
        n_val = 1
        n_train = n - 2

    train_ids = set(shuffled[:n_train])
    val_ids = set(shuffled[n_train:n_train + n_val])
    test_ids = set(shuffled[n_train + n_val:])

    pairs["split"] = pairs["id"].astype(str).apply(
        lambda x: (
            "train"
            if x in train_ids
            else ("validation" if x in val_ids else "test")
        )
    )

    split_df = pairs[["id", "split"]].copy()
    split_df.to_csv(SPLIT_FILE, index=False)
    print("Created source split:", SPLIT_FILE)

print("Selected recordings:", len(pairs))
print("Recording split:", pairs["split"].value_counts().to_dict())

display(
    pairs[
        [
            "id",
            "classification",
            "split",
            "duration_sec",
            "sample_rate",
        ]
    ].head(20)
)


## 7. Load the Whisper processor from the existing trained model and define sentence-aware preprocessing

The preprocessing logic remains the same as the current training notebook.

The `transcript` column from `conversations.csv` is always the ground-truth label.


In [ ]:
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    WhisperConfig,
)
from tqdm.auto import tqdm
from copy import deepcopy

processor = WhisperProcessor.from_pretrained(
    str(EXISTING_MODEL),
    language="Hindi",
    task="transcribe"
)
processor.feature_extractor.return_attention_mask = True
base_config = WhisperConfig.from_pretrained(
    str(EXISTING_MODEL)
)

def configure_generation(model):
    for key in ("max_length", "suppress_tokens", "begin_suppress_tokens", "forced_decoder_ids"):
        value = getattr(model.config, key, None)
        if value is not None and getattr(model.generation_config, key, None) is None:
            setattr(model.generation_config, key, deepcopy(value))
        setattr(model.config, key, None)

    model.generation_config.max_length = model.config.max_target_positions
    model.generation_config.language = "hindi"
    model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None
    model.generation_config.return_timestamps = False
    model.generation_config.num_beams = 1
    model.generation_config.do_sample = False
    model.generation_config.use_cache = False
    model.config.use_cache = False
    model.gradient_checkpointing_disable()
    return model


def token_count(text):
    return len(
        processor.tokenizer(
            text,
            add_special_tokens=True
        ).input_ids
    )


# ------------------------------------------------------------
# Ground-truth sentence / clause detection
# ------------------------------------------------------------

ABBREVIATIONS = {
    "dr.", "mr.", "mrs.", "ms.", "prof.", "etc.",
    "e.g.", "i.e.", "vs.", "mg.", "ml.", "tab.",
    "cap.", "inj.", "no.", "fig.", "sec.", "approx."
}


def _strip_closing_punctuation(word):
    return word.rstrip('"\'”’»)]}')


def ends_sentence(word):
    w = _strip_closing_punctuation(str(word))
    lw = w.lower()

    if w.endswith(("।", "॥", "?", "!")):
        return True

    if w.endswith("..."):
        return True

    if not w.endswith("."):
        return False

    if lw in ABBREVIATIONS:
        return False

    # Initials such as A. / B. should not split a sentence.
    if re.fullmatch(r"(?:[A-Za-z]\.)+", w):
        return False

    return True



def ends_clause(word):
    w = _strip_closing_punctuation(str(word))
    return (
        ends_sentence(w)
        or w.endswith((",", ";", ":", "—", "–"))
    )



def word_ranges_by_sentence(words):
    ranges = []
    start = 0

    for i, word in enumerate(words, 1):
        if ends_sentence(word):
            ranges.append((start, i))
            start = i

    if start < len(words):
        ranges.append((start, len(words)))

    return ranges



def split_range_by_clauses(words, start, end):
    boundaries = []

    for i in range(start + 1, end):
        if ends_clause(words[i - 1]):
            boundaries.append(i)

    boundaries.append(end)

    units = []
    cursor = start

    for boundary in boundaries:
        if boundary <= cursor:
            continue

        units.append((cursor, boundary))
        cursor = boundary

    return units


# ------------------------------------------------------------
# Fast audio-boundary estimation
# ------------------------------------------------------------

def estimate_word_boundaries(words, duration):
    """
    Estimate the time of every transcript-word boundary from text weight.

    This is deliberately deterministic and fast. The weights approximate
    TTS speaking time using the number of alphanumeric/Devanagari characters.
    Actual sentence boundaries are later snapped toward nearby quiet points.
    """

    if not words:
        return [0.0]

    weights = []
    for word in words:
        clean = re.sub(r"[^0-9A-Za-z\u0900-\u097F]", "", str(word))
        weights.append(max(1, len(clean)))

    cumulative = np.concatenate(([0.0], np.cumsum(weights).astype(float)))
    total = float(cumulative[-1])

    if total <= 0:
        return np.linspace(0.0, duration, len(words) + 1).tolist()

    boundaries = (cumulative / total * duration).tolist()
    boundaries[0] = 0.0
    boundaries[-1] = float(duration)
    return boundaries


def quiet_boundary(audio, sr, target_sec, search_radius=1.25):
    """Find a nearby low-energy point; fall back to target when uncertain."""

    duration = len(audio) / sr

    if target_sec <= 0.0 or target_sec >= duration:
        return float(min(max(target_sec, 0.0), duration)), 0.0

    start = max(0.05, target_sec - search_radius)
    end = min(duration - 0.05, target_sec + search_radius)

    if end <= start:
        return float(target_sec), 0.0

    frame = max(1, int(0.08 * sr))
    hop = max(1, int(0.02 * sr))
    start_i = int(start * sr)
    end_i = min(len(audio), int(end * sr))

    if end_i - start_i < frame:
        return float(target_sec), 0.0

    samples = audio[start_i:end_i].astype(np.float32, copy=False)
    n_frames = 1 + max(0, (len(samples) - frame) // hop)

    rms = np.empty(n_frames, dtype=np.float32)
    times = np.empty(n_frames, dtype=np.float32)

    for i in range(n_frames):
        a = i * hop
        b = a + frame
        x = samples[a:b]
        rms[i] = np.sqrt(np.mean(x * x) + 1e-10)
        times[i] = (start_i + a + frame / 2) / sr

    # Bias slightly toward the requested boundary so a random deep pause
    # elsewhere in the search window is not selected.
    normalized = rms / max(float(rms.max()), 1e-8)
    distance = np.abs(times - target_sec) / max(search_radius, 1e-6)
    score = normalized + 0.20 * distance
    idx = int(np.argmin(score))
    chosen = float(times[idx])

    return chosen, abs(chosen - target_sec)


def refine_sentence_boundary_times(audio, sr, words, raw_boundaries):
    """Snap sentence-ending word boundaries toward nearby low-energy points."""

    sentence_ranges = word_ranges_by_sentence(words)
    boundary_times = list(raw_boundaries)
    snapped = 0
    max_adjustment = 0.0

    for start, end in sentence_ranges[:-1]:
        target = raw_boundaries[end]
        chosen, adjustment = quiet_boundary(
            audio,
            sr,
            target
        )

        # Never allow a snapped boundary to cross neighboring raw boundaries.
        lower = raw_boundaries[start] + 0.05
        upper = raw_boundaries[min(len(words), end + max(1, len(words) // 100))]
        chosen = min(max(chosen, lower), upper)

        if chosen > 0 and chosen < (len(audio) / sr):
            boundary_times[end] = chosen
            snapped += 1
            max_adjustment = max(max_adjustment, adjustment)

    # Enforce strict monotonicity after snapping.
    for i in range(1, len(boundary_times)):
        if boundary_times[i] <= boundary_times[i - 1]:
            boundary_times[i] = boundary_times[i - 1] + 1.0 / sr

    boundary_times[0] = 0.0
    boundary_times[-1] = len(audio) / sr

    return boundary_times, snapped, max_adjustment


# ------------------------------------------------------------
# Build atomic units, then pack them into Whisper-safe chunks.
# ------------------------------------------------------------

def greedy_word_units(words, start, end, boundary_times):
    units = []
    cursor = start

    while cursor < end:
        best = None

        for candidate in range(cursor + 1, end + 1):
            text = " ".join(words[cursor:candidate])
            labels = token_count(text)
            duration_part = boundary_times[candidate] - boundary_times[cursor]

            if labels <= MAX_LABEL_TOKENS and duration_part <= MAX_AUDIO_SEC:
                best = candidate
            else:
                break

        if best is None:
            # Extremely unlikely: even one word cannot fit.
            one = " ".join(words[cursor:cursor + 1])
            raise RuntimeError(
                f"Single word cannot fit safe limits: {one!r}"
            )

        units.append({
            "start_word": cursor,
            "end_word": best,
            "level": "word_fallback",
        })
        cursor = best

    return units


def make_atomic_units(words, boundary_times):
    units = []

    for sentence_start, sentence_end in word_ranges_by_sentence(words):
        sentence_text = " ".join(words[sentence_start:sentence_end])
        sentence_tokens = token_count(sentence_text)
        sentence_duration = (
            boundary_times[sentence_end]
            - boundary_times[sentence_start]
        )

        if (
            sentence_tokens <= MAX_LABEL_TOKENS
            and sentence_duration <= MAX_AUDIO_SEC
        ):
            units.append({
                "start_word": sentence_start,
                "end_word": sentence_end,
                "level": "sentence",
            })
            continue

        # A sentence is too large. Try clause boundaries first.
        clause_ranges = split_range_by_clauses(
            words,
            sentence_start,
            sentence_end
        )

        for clause_start, clause_end in clause_ranges:
            clause_text = " ".join(words[clause_start:clause_end])
            clause_tokens = token_count(clause_text)
            clause_duration = (
                boundary_times[clause_end]
                - boundary_times[clause_start]
            )

            if (
                clause_tokens <= MAX_LABEL_TOKENS
                and clause_duration <= MAX_AUDIO_SEC
            ):
                units.append({
                    "start_word": clause_start,
                    "end_word": clause_end,
                    "level": "clause",
                })
            else:
                units.extend(
                    greedy_word_units(
                        words,
                        clause_start,
                        clause_end,
                        boundary_times
                    )
                )

    return units


def pack_units_into_chunks(words, units, boundary_times):
    chunks = []
    current = None

    for unit in units:
        ustart = unit["start_word"]
        uend = unit["end_word"]
        utext = " ".join(words[ustart:uend]).strip()

        if current is None:
            current = {
                "start_word": ustart,
                "end_word": uend,
                "level": unit["level"],
            }
            continue

        candidate_start = current["start_word"]
        candidate_end = uend
        candidate_text = " ".join(words[candidate_start:candidate_end]).strip()
        candidate_tokens = token_count(candidate_text)
        candidate_duration = (
            boundary_times[candidate_end]
            - boundary_times[candidate_start]
        )

        if (
            candidate_tokens <= MAX_LABEL_TOKENS
            and candidate_duration <= MAX_AUDIO_SEC
        ):
            current["end_word"] = candidate_end
            if unit["level"] != "sentence" or current["level"] != "sentence":
                current["level"] = (
                    "sentence"
                    if current["level"] == "sentence" and unit["level"] == "sentence"
                    else "mixed"
                )
        else:
            chunks.append(current)
            current = {
                "start_word": ustart,
                "end_word": uend,
                "level": unit["level"],
            }

    if current is not None:
        chunks.append(current)

    return chunks


def make_chunks(audio, sr, transcript):
    words = transcript.split()

    if not words:
        raise RuntimeError("Empty transcript after splitting.")

    duration = len(audio) / sr

    raw_boundaries = estimate_word_boundaries(
        words,
        duration
    )

    # Only normal sentence boundaries are snapped to low-energy points.
    boundary_times, snapped_count, max_adjustment = refine_sentence_boundary_times(
        audio,
        sr,
        words,
        raw_boundaries
    )

    atomic_units = make_atomic_units(
        words,
        boundary_times
    )

    chunks = pack_units_into_chunks(
        words,
        atomic_units,
        boundary_times
    )

    if not chunks:
        raise RuntimeError("No chunks were produced.")

    parts = []
    for chunk in chunks:
        a = chunk["start_word"]
        b = chunk["end_word"]
        text = " ".join(words[a:b]).strip()
        start_sec = 0.0 if a == 0 else float(boundary_times[a])
        end_sec = duration if b == len(words) else float(boundary_times[b])

        parts.append({
            "start": start_sec,
            "end": end_sec,
            "text": text,
            "boundary_level": chunk["level"],
        })

    # Complete text coverage: every transcript word exactly once.
    joined = " ".join(p["text"] for p in parts).split()
    if joined != words:
        raise RuntimeError(
            "Transcript coverage check failed: chunk labels do not reproduce the source transcript exactly."
        )

    # Complete audio coverage: no gaps and no missing tail.
    previous_end = 0.0
    for i, part in enumerate(parts):
        if part["start"] < previous_end - 1e-6:
            raise RuntimeError(f"Audio overlap detected between chunks {i-1} and {i}.")
        if part["start"] > previous_end + (1.5 / sr):
            raise RuntimeError(f"Audio gap detected before chunk {i}.")
        if not (0.0 <= part["start"] < part["end"] <= duration + 1e-6):
            raise RuntimeError(f"Invalid audio bounds in chunk {i}.")
        previous_end = part["end"]

    if abs(previous_end - duration) > 1.5 / sr:
        raise RuntimeError(
            f"Audio tail coverage failed: final chunk ends at {previous_end:.4f}s of {duration:.4f}s."
        )

    return parts, snapped_count, max_adjustment

print("Sentence-aware preprocessing functions ready.")


## 8. Automatic preprocessing: sentence-aware split, full-coverage validation, save, resume

This stage is designed so that a normal long recording is **not discarded** merely because it exceeds Whisper's input limits. It is partitioned into complete-sentence chunks whenever possible. Only an oversized individual sentence/clause falls back to a finer boundary. Any true source failure stops the pipeline before training.


In [ ]:
# ============================================================
# PERSISTENT PREPROCESSING OUTPUTS — SENTENCE-SAFE V2
# ============================================================

PREP_ROOT = NEW_DATA_ROOT / "preprocessing_sentence_safe_v2"
SEGMENT_DIR = PREP_ROOT / "segments"
MANIFEST = PREP_ROOT / "full_segments_manifest.csv"
REPORT_FILE = PREP_ROOT / "full_preprocessing_report.csv"

PREP_ROOT.mkdir(parents=True, exist_ok=True)
SEGMENT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_COLUMNS = [
    "segment_id", "original_id", "split", "audio_path", "transcript",
    "start_sec", "end_sec", "duration_sec", "original_duration_sec",
    "label_tokens", "boundary_level",
]

REPORT_COLUMNS = [
    "original_id", "split", "status", "reason", "num_chunks",
    "duration_sec", "transcript_tokens", "text_coverage",
    "audio_coverage", "timing_method", "sentence_boundaries_snapped",
    "max_boundary_adjustment_sec", "chunk_boundary_mode",
]

if MANIFEST.exists():
    segment_df_existing = pd.read_csv(MANIFEST)
else:
    segment_df_existing = pd.DataFrame(columns=MANIFEST_COLUMNS)

if REPORT_FILE.exists():
    report_df_existing = pd.read_csv(REPORT_FILE)
else:
    report_df_existing = pd.DataFrame(columns=REPORT_COLUMNS)

segment_records_by_id = {}
for rec in segment_df_existing.to_dict("records"):
    segment_records_by_id.setdefault(
        str(rec["original_id"]), []
    ).append(rec)

report_by_id = {}
for rec in report_df_existing.to_dict("records"):
    report_by_id[str(rec["original_id"])] = rec


def accepted_state_is_valid(sample_id):
    recs = segment_records_by_id.get(sample_id, [])
    report = report_by_id.get(sample_id)

    if not recs or not report:
        return False

    if report.get("status") != "accepted":
        return False

    if report.get("text_coverage") != "pass":
        return False

    if report.get("audio_coverage") != "pass":
        return False

    # Every chunk must still exist on Drive.
    return all(
        Path(str(r["audio_path"])).exists()
        for r in recs
    )


def remove_existing_record(sample_id):
    segment_records_by_id.pop(sample_id, None)

    for p in SEGMENT_DIR.glob(f"{sample_id}_sentencesafe_*.wav"):
        try:
            p.unlink()
        except FileNotFoundError:
            pass


def persist_preprocessing_state():
    flat = [
        r
        for recs in segment_records_by_id.values()
        for r in recs
    ]

    if flat:
        segment_df = pd.DataFrame(flat)
        segment_df = segment_df[
            MANIFEST_COLUMNS
        ].sort_values(
            ["original_id", "segment_id"]
        ).reset_index(drop=True)
    else:
        segment_df = pd.DataFrame(
            columns=MANIFEST_COLUMNS
        )

    segment_df.to_csv(
        MANIFEST,
        index=False
    )

    report_state = pd.DataFrame(
        list(report_by_id.values())
    )

    if not report_state.empty:
        report_state = report_state[
            REPORT_COLUMNS
        ].sort_values(
            "original_id"
        ).reset_index(drop=True)

    report_state.to_csv(
        REPORT_FILE,
        index=False
    )

    return segment_df, report_state


# Keep the live DataFrames defined for the entire notebook.
segment_df, report_df = persist_preprocessing_state()

preprocess_started = time.perf_counter()

for row in tqdm(
    pairs.itertuples(),
    total=len(pairs),
    desc="Preprocessing originals",
    unit="recording"
):

    sample_id = str(row.id)
    split = str(row.split)
    source_text = str(row.transcript).strip()

    # Reuse fully validated previous work after a Colab restart.
    if accepted_state_is_valid(sample_id):
        continue

    # Retry failed sources on a later resume; they must ultimately be fixed
    # or the training gate will stop before any training starts.
    remove_existing_record(sample_id)

    try:

        if not source_text:
            raise RuntimeError("Empty transcript.")

        original_tokens = token_count(source_text)

        audio, sr = sf.read(
            str(row.audio_path),
            dtype="float32"
        )

        if audio.ndim > 1:
            audio = np.mean(
                audio,
                axis=1
            )

        if len(audio) == 0 or not np.isfinite(audio).all():
            raise RuntimeError(
                "Empty or non-finite audio."
            )

        if sr != 16000:
            gcd = int(np.gcd(sr, 16000))
            audio = resample_poly(
                audio,
                16000 // gcd,
                sr // gcd
            ).astype(np.float32)
            sr = 16000

        duration = len(audio) / 16000

        if not math.isfinite(duration) or duration <= 0:
            raise RuntimeError(
                "Invalid audio duration after normalization."
            )

        # Fast path: recording already fits Whisper-safe limits.
        if (
            duration <= MAX_AUDIO_SEC
            and original_tokens <= MAX_LABEL_TOKENS
        ):

            parts = [{
                "start": 0.0,
                "end": duration,
                "text": source_text,
                "boundary_level": "whole_recording",
            }]

            snapped_count = 0
            max_adjustment = 0.0
            timing_method = "whole_recording"

        else:

            parts, snapped_count, max_adjustment = make_chunks(
                audio,
                16000,
                source_text
            )

            timing_method = "text_weight_plus_quiet_point"

        # ----------------------------------------------------
        # Validate complete transcript coverage BEFORE writing
        # training state.
        # ----------------------------------------------------

        source_words = source_text.split()
        joined_words = " ".join(
            p["text"] for p in parts
        ).split()

        if joined_words != source_words:
            raise RuntimeError(
                "Transcript coverage failed: source words and chunk words differ."
            )

        # ----------------------------------------------------
        # Write chunks, with no silent dropping.
        # ----------------------------------------------------

        segment_rows = []

        for index, part in enumerate(parts):

            start_sample = max(
                0,
                min(
                    len(audio),
                    int(round(part["start"] * 16000))
                )
            )

            end_sample = max(
                0,
                min(
                    len(audio),
                    int(round(part["end"] * 16000))
                )
            )

            if end_sample <= start_sample:
                raise RuntimeError(
                    f"Chunk {index} is empty after sample conversion."
                )

            clip = audio[start_sample:end_sample]
            clip_duration = len(clip) / 16000
            part_text = str(part["text"]).strip()
            label_tokens = token_count(part_text)

            if not (0 < clip_duration <= MAX_AUDIO_SEC):
                raise RuntimeError(
                    f"Chunk {index} is {clip_duration:.3f}s; limit is {MAX_AUDIO_SEC}s."
                )

            if not (0 < label_tokens <= MAX_LABEL_TOKENS):
                raise RuntimeError(
                    f"Chunk {index} has {label_tokens} tokens; limit is {MAX_LABEL_TOKENS}."
                )

            segment_id = (
                f"{sample_id}_sentencesafe_{index:03d}"
            )

            out = (
                SEGMENT_DIR /
                f"{segment_id}.wav"
            )

            sf.write(
                str(out),
                clip,
                16000,
                subtype="PCM_16"
            )

            segment_rows.append({
                "segment_id": segment_id,
                "original_id": sample_id,
                "split": split,
                "audio_path": str(out),
                "transcript": part_text,
                "start_sec": start_sample / 16000,
                "end_sec": end_sample / 16000,
                "duration_sec": clip_duration,
                "original_duration_sec": duration,
                "label_tokens": label_tokens,
                "boundary_level": part["boundary_level"],
            })

        if not segment_rows:
            raise RuntimeError(
                "No chunks were created."
            )

        # ----------------------------------------------------
        # Final audio coverage validation from the written rows.
        # ----------------------------------------------------

        previous_end = 0.0

        for i, rec in enumerate(segment_rows):

            start_sec = float(rec["start_sec"])
            end_sec = float(rec["end_sec"])

            if start_sec < previous_end - (1.5 / 16000):
                raise RuntimeError(
                    f"Audio overlap detected around chunk {i}."
                )

            if start_sec > previous_end + (1.5 / 16000):
                raise RuntimeError(
                    f"Audio gap detected before chunk {i}."
                )

            previous_end = end_sec

        if abs(previous_end - duration) > (1.5 / 16000):
            raise RuntimeError(
                "Audio coverage failed: final chunk does not reach recording end."
            )

        # ----------------------------------------------------
        # Accept ONLY after all chunks pass all checks.
        # ----------------------------------------------------

        segment_records_by_id[sample_id] = segment_rows

        levels = [
            r["boundary_level"]
            for r in segment_rows
        ]

        if all(level in {"sentence", "whole_recording"} for level in levels):
            boundary_mode = "sentence_only"
        elif any(level == "word_fallback" for level in levels):
            boundary_mode = "word_fallback_used"
        else:
            boundary_mode = "clause_fallback_used"

        report_by_id[sample_id] = {
            "original_id": sample_id,
            "split": split,
            "status": "accepted",
            "reason": "",
            "num_chunks": len(segment_rows),
            "duration_sec": duration,
            "transcript_tokens": original_tokens,
            "text_coverage": "pass",
            "audio_coverage": "pass",
            "timing_method": timing_method,
            "sentence_boundaries_snapped": int(snapped_count),
            "max_boundary_adjustment_sec": float(max_adjustment),
            "chunk_boundary_mode": boundary_mode,
        }

    except Exception as exc:

        # No partial chunks from a failed original are allowed to remain
        # in the training manifest.
        remove_existing_record(sample_id)

        report_by_id[sample_id] = {
            "original_id": sample_id,
            "split": split,
            "status": "failed",
            "reason": str(exc),
            "num_chunks": 0,
            "duration_sec": float(getattr(row, "duration_sec", 0.0)),
            "transcript_tokens": token_count(source_text) if source_text else 0,
            "text_coverage": "fail",
            "audio_coverage": "fail",
            "timing_method": "failed",
            "sentence_boundaries_snapped": 0,
            "max_boundary_adjustment_sec": 0.0,
            "chunk_boundary_mode": "none",
        }

        print(
            f"\nFAILED {sample_id}: {exc}",
            flush=True
        )

    # Save after every original. A Colab disconnect therefore does not
    # destroy already-processed recordings.
    # Persist and immediately refresh the live DataFrames.
    segment_df, report_df = persist_preprocessing_state()

PREPARATION_SECONDS = (
    time.perf_counter() - preprocess_started
)

# segment_df and report_df are maintained live above.
# No *_state variables are required.

print("\n" + "=" * 70)
print("AUTOMATIC PREPROCESSING COMPLETE")
print("=" * 70)
print(
    "Elapsed:",
    round(PREPARATION_SECONDS / 60, 2),
    "minutes"
)
print(
    "Selected originals:",
    len(pairs)
)
print(
    "Accepted originals:",
    int((report_df["status"] == "accepted").sum())
)
print(
    "Failed originals:",
    int((report_df["status"] == "failed").sum())
)
print(
    "Prepared chunks:",
    len(segment_df)
)
print(
    "Manifest:",
    MANIFEST
)
print(
    "Report:",
    REPORT_FILE
)


## 9. Automatic preprocessing gate — zero silent record loss

The notebook does not ask for manual approval. It does, however, enforce a **hard completeness gate**: if any selected source recording is failed, missing from the report, missing from the segment manifest, or has failed text/audio coverage checks, the notebook stops **before dataset preparation and training**.


In [ ]:
# ============================================================
# AUTOMATIC PREPROCESSING COMPLETENESS GATE
# ============================================================

if report_df.empty:
    raise RuntimeError(
        "No preprocessing report exists."
    )

selected_ids = set(
    pairs["id"].astype(str)
)
report_ids = set(
    report_df["original_id"].astype(str)
)

if report_ids != selected_ids:
    missing_report = sorted(
        selected_ids - report_ids
    )
    extra_report = sorted(
        report_ids - selected_ids
    )

    raise RuntimeError(
        "Preprocessing report does not exactly match the selected source recordings. "
        f"Missing: {missing_report[:20]}; Extra: {extra_report[:20]}"
    )

failed_rows = report_df[
    report_df["status"].astype(str) != "accepted"
].copy()

if not failed_rows.empty:

    print("=" * 70)
    print("PREPROCESSING FAILED — TRAINING BLOCKED")
    print("=" * 70)

    print(
        "Failed recordings:",
        len(failed_rows)
    )

    display(
        failed_rows[
            [
                "original_id",
                "split",
                "reason",
                "text_coverage",
                "audio_coverage",
            ]
        ].head(50)
    )

    raise RuntimeError(
        "At least one selected recording failed preprocessing. "
        "No training data will be used until preprocessing reaches 100% acceptance."
    )

accepted_ids = set(
    report_df["original_id"].astype(str)
)

# Always define this variable before any later training/export cell uses it.
# Under a passing gate it is an empty set.
failed_ids = selected_ids - accepted_ids

if accepted_ids != selected_ids:
    raise RuntimeError(
        "Accepted-record set does not exactly match the selected source set."
    )

if segment_df.empty:
    raise RuntimeError(
        "No chunks are available after preprocessing."
    )

segment_ids = set(
    segment_df["original_id"].astype(str)
)

if segment_ids != selected_ids:
    missing_segments = sorted(
        selected_ids - segment_ids
    )

    raise RuntimeError(
        "Some accepted recordings have no chunks. "
        f"Missing chunked IDs: {missing_segments[:20]}"
    )

# ------------------------------------------------------------
# Per-chunk safety checks
# ------------------------------------------------------------

if not (
    segment_df["duration_sec"] > 0
).all():
    raise RuntimeError(
        "A prepared chunk has zero or negative duration."
    )

if not (
    segment_df["duration_sec"] <= MAX_AUDIO_SEC
).all():
    raise RuntimeError(
        "A prepared chunk exceeds the audio safety limit."
    )

if not (
    segment_df["label_tokens"] > 0
).all():
    raise RuntimeError(
        "A prepared chunk has an empty label."
    )

if not (
    segment_df["label_tokens"] <= MAX_LABEL_TOKENS
).all():
    raise RuntimeError(
        "A prepared label exceeds the token safety limit."
    )

# ------------------------------------------------------------
# Verify exact transcript coverage per original.
# ------------------------------------------------------------

source_text_map = dict(
    zip(
        pairs["id"].astype(str),
        pairs["transcript"].astype(str)
    )
)

for sample_id, group in segment_df.groupby(
    "original_id",
    sort=True
):

    actual = " ".join(
        group.sort_values("segment_id")["transcript"].tolist()
    ).split()

    expected = source_text_map[str(sample_id)].split()

    if actual != expected:
        raise RuntimeError(
            f"Transcript coverage mismatch for {sample_id}. "
            "The notebook will not train on partially represented text."
        )

# ------------------------------------------------------------
# Verify exact audio coverage per original.
# ------------------------------------------------------------

source_duration_map = dict(
    zip(
        pairs["id"].astype(str),
        pairs["duration_sec"].astype(float)
    )
)

for sample_id, group in segment_df.groupby(
    "original_id",
    sort=True
):

    ordered = group.sort_values(
        "segment_id"
    )

    previous_end = 0.0

    for row in ordered.itertuples():
        start_sec = float(row.start_sec)
        end_sec = float(row.end_sec)

        if start_sec > previous_end + 0.002:
            raise RuntimeError(
                f"Audio gap detected for {sample_id}."
            )

        if start_sec < previous_end - 0.002:
            raise RuntimeError(
                f"Audio overlap detected for {sample_id}."
            )

        previous_end = end_sec

    expected_duration = source_duration_map[str(sample_id)]

    if abs(previous_end - expected_duration) > 0.002:
        raise RuntimeError(
            f"Audio tail coverage mismatch for {sample_id}: "
            f"chunks end at {previous_end:.4f}s, source is {expected_duration:.4f}s."
        )

print("=" * 70)
print("PREPROCESSING COMPLETE — NO RECORDING LOSS")
print("=" * 70)
print("Selected originals:", len(selected_ids))
print("Accepted originals:", len(accepted_ids))
print("Failed originals:", len(failed_rows))
print("Accepted chunks:", len(segment_df))
print(
    "Chunk duration max:",
    round(float(segment_df["duration_sec"].max()), 2),
    "sec"
)
print(
    "Label token max:",
    int(segment_df["label_tokens"].max())
)
print("\nChunks by split:")
print(
    segment_df["split"].value_counts().to_dict()
)

print("\nBoundary modes:")
print(
    report_df["chunk_boundary_mode"].value_counts().to_dict()
)

print(
    "\nPreprocessing report:",
    REPORT_FILE
)

print(
    "Manifest:",
    MANIFEST
)

print(
    "\nTraining gate PASSED. All selected recordings are represented completely."
)


## 10. Build Hugging Face datasets from accepted chunks

In [ ]:
from datasets import Dataset, DatasetDict


def validate_features(features, mask, label_ids, segment_id):
    features = np.asarray(features)
    mask = np.asarray(mask)
    if features.ndim != 2:
        raise RuntimeError(f"{segment_id}: feature tensor must be 2D.")
    if mask.shape != (features.shape[-1],):
        raise RuntimeError(f"{segment_id}: audio attention mask shape mismatch.")
    if not np.isfinite(features).all():
        raise RuntimeError(f"{segment_id}: non-finite input features.")
    if not np.isin(mask, [0, 1]).all() or not mask.any():
        raise RuntimeError(f"{segment_id}: invalid/empty audio attention mask.")
    if not 0 < len(label_ids) <= MAX_LABEL_TOKENS:
        raise RuntimeError(f"{segment_id}: invalid label length {len(label_ids)}.")


def prepare_split(df_part, split):
    records = []
    for row in tqdm(
        df_part.itertuples(),
        total=len(df_part),
        desc=f"Features: {split}",
        unit="chunk"
    ):
        audio, sr = sf.read(row.audio_path, dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != 16000 or not 0 < len(audio) <= int(MAX_AUDIO_SEC * 16000):
            raise RuntimeError(
                f"{row.segment_id}: expected mono 16 kHz chunk under {MAX_AUDIO_SEC}s."
            )

        features = processor.feature_extractor(
            audio,
            sampling_rate=16000,
            padding="max_length",
            truncation=False,
            return_attention_mask=True,
            return_tensors="np"
        )

        label_ids = processor.tokenizer(
            row.transcript,
            add_special_tokens=True
        ).input_ids

        mel = features.input_features[0]
        mask = features.attention_mask[0]
        validate_features(mel, mask, label_ids, row.segment_id)

        records.append({
            "input_features": mel,
            "attention_mask": mask,
            "labels": label_ids,
        })

    return Dataset.from_list(records)


processed = DatasetDict({
    split: prepare_split(
        segment_df[segment_df["split"] == split],
        split
    )
    for split in ("train", "validation", "test")
})

for split in processed:
    if len(processed[split]) == 0:
        raise RuntimeError(f"Split {split} has no accepted chunks.")

print(processed)
print("Audio attention masks are stored in all splits.")


## 11. Data collator + WER

In [ ]:
from jiwer import wer
from torch.nn.utils.rnn import pad_sequence


class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, decoder_start_token_id):
        self.decoder_start_token_id = decoder_start_token_id

    def __call__(self, features):
        inputs = torch.stack([
            torch.as_tensor(
                f["input_features"],
                dtype=torch.float32
            )
            for f in features
        ])

        masks = torch.stack([
            torch.as_tensor(
                f["attention_mask"],
                dtype=torch.long
            )
            for f in features
        ])

        labels = []
        for f in features:
            ids = list(f["labels"])
            if not ids or ids[0] != self.decoder_start_token_id:
                raise RuntimeError(
                    "Labels must begin with decoder_start_token_id."
                )
            labels.append(
                torch.tensor(
                    ids[1:],
                    dtype=torch.long
                )
            )

        return {
            "input_features": inputs,
            "attention_mask": masks,
            "labels": pad_sequence(
                labels,
                batch_first=True,
                padding_value=-100
            )
        }


def compute_metrics(pred):
    pred_ids = pred.predictions
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    predicted = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    references = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    return {
        "wer": 100 * wer(references, predicted)
    }


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    base_config.decoder_start_token_id
)

probe = data_collator([processed["train"][0]])
print(
    "Batch shapes:",
    {k: tuple(v.shape) for k, v in probe.items()}
)
print(
    "Unmasked audio frames:",
    int(probe["attention_mask"].sum().item())
)
del probe


## 12. Create a resumable incremental Whisper Small trainer

This trainer starts from the existing trained model rather than the original `openai/whisper-small`.

Starting model:

```text
MyDrive/medical-whisper/final_whisper-small_medical_full
```

Incremental checkpoints:

```text
MyDrive/medical-whisper/conversation_dataset/incremental_training/
```

The original trained model remains unchanged.


In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

set_seed(RANDOM_SEED)
torch.cuda.empty_cache()

FULL_RUN_ROOT = NEW_DATA_ROOT / "incremental_training_v2"
TRAIN_OUTPUT = FULL_RUN_ROOT / "training"
FULL_RUN_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_OUTPUT.mkdir(parents=True, exist_ok=True)

# Fresh base model object. Trainer state determines whether we resume.
print("Loading existing trained Whisper Small model...", flush=True)
print("Starting model:", EXISTING_MODEL, flush=True)
model = configure_generation(
    WhisperForConditionalGeneration.from_pretrained(
        str(EXISTING_MODEL)
    )
)
model.config.use_cache = False
model.generation_config.use_cache = False
model.gradient_checkpointing_disable()

# Use a stable output directory so checkpoints survive Colab runtime restarts.
last_checkpoint = (
    get_last_checkpoint(str(TRAIN_OUTPUT))
    if RESUME_TRAINING
    else None
)

if last_checkpoint:
    print("Resumable checkpoint found:", last_checkpoint, flush=True)
else:
    print("No checkpoint will be resumed. Training will start from the existing v1 fine-tuned model.", flush=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(TRAIN_OUTPUT),
    num_train_epochs=NUM_EPOCHS,
    max_steps=-1,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="linear",
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    logging_nan_inf_filter=False,
    disable_tqdm=False,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=model.config.max_target_positions,
    generation_num_beams=1,
    fp16=True,
    gradient_checkpointing=False,
    report_to="none",
    seed=RANDOM_SEED,
    data_seed=RANDOM_SEED,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)


class ProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        print(
            f"\nTRAINING START: {state.max_steps if state.max_steps > 0 else 'epoch-based'} scheduled steps",
            flush=True
        )
        print(
            f"Train chunks: {len(processed['train'])} | "
            f"Validation chunks: {len(processed['validation'])} | "
            f"Test chunks: {len(processed['test'])}",
            flush=True
        )

    def on_save(self, args, state, control, **kwargs):
        print(
            f"Checkpoint saved at global step {state.global_step}",
            flush=True
        )

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            print(
                f"Validation at step {state.global_step}: {metrics}",
                flush=True
            )


progress_callback = ProgressCallback()

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed["train"],
    eval_dataset=processed["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[progress_callback],
)

print("\n" + "=" * 70)
print("TRAINER READY")
print("=" * 70)
print("Starting model:", EXISTING_MODEL)
print("Training output:", TRAIN_OUTPUT)
print("Resume checkpoint:", last_checkpoint)
print("Epochs:", NUM_EPOCHS)
print("Train batch size:", TRAIN_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Evaluation steps:", EVAL_STEPS)
print("Checkpoint steps:", SAVE_STEPS)
print("Gradient checkpointing:", training_args.gradient_checkpointing)
print("FP16:", training_args.fp16)


## 13. Start incremental training

By default this notebook starts a **fresh v2 run** from `final_whisper-small_medical_full`.

Set `RESUME_TRAINING = True` in the configuration only when intentionally continuing an interrupted v2 run from the latest checkpoint under `incremental_training_v2/training`.


In [ ]:
# ============================================================
# FULL TRAINING
# ============================================================

if set(processed) != {"train", "validation", "test"}:
    raise RuntimeError("Expected train/validation/test datasets.")

if trainer.args.device.type != "cuda":
    raise RuntimeError("Select a Colab GPU runtime.")

if not len(processed["train"]):
    raise RuntimeError("No accepted training chunks.")

model.gradient_checkpointing_disable()
model.config.use_cache = False
model.generation_config.use_cache = False
model.zero_grad(set_to_none=True)
model.train()

print("=" * 70)
print("STARTING FULL WHISPER SMALL TRAINING")
print("=" * 70)
print("Source recordings selected:", len(pairs))
print("Accepted recordings:", len(accepted_ids))
print("Failed recordings:", len(failed_ids))
print("Training chunks:", len(processed["train"]))
print("Validation chunks:", len(processed["validation"]))
print("Test chunks:", len(processed["test"]))
print("Epochs:", NUM_EPOCHS)
print("Resume from:", last_checkpoint if last_checkpoint else "NONE — fresh run from existing v1 model")

training_started = time.perf_counter()
train_result = trainer.train(
    resume_from_checkpoint=last_checkpoint
)
torch.cuda.synchronize()
training_seconds = time.perf_counter() - training_started

print("\n" + "=" * 70)
print("FULL TRAINING COMPLETE")
print("=" * 70)
print("Training seconds:", round(training_seconds, 1))
print("Training metrics:", train_result.metrics)


## 14. Final validation, test prediction, and export

The test set is the held-out portion of the new conversation dataset.

The original `final_whisper-small_medical_full` is never overwritten.
The new model is exported separately as `final_whisper-small_medical_full_v2`.


In [ ]:
# ============================================================
# FINAL EVALUATION + EXPORT
# ============================================================

if trainer.state.global_step <= 0:
    raise RuntimeError("Training did not complete any optimizer steps.")

model.gradient_checkpointing_disable()
model.config.use_cache = True
model.generation_config.use_cache = True

print("Running final validation...", flush=True)
validation_started = time.perf_counter()
validation_results = trainer.evaluate(
    eval_dataset=processed["validation"],
    metric_key_prefix="validation"
)
validation_seconds = time.perf_counter() - validation_started

print("Running final test prediction...", flush=True)
test_started = time.perf_counter()
pred = trainer.predict(
    processed["test"],
    metric_key_prefix="test"
)
test_seconds = time.perf_counter() - test_started

pred_ids = pred.predictions
if isinstance(pred_ids, tuple):
    pred_ids = pred_ids[0]

predicted_text = processor.tokenizer.batch_decode(
    pred_ids,
    skip_special_tokens=True
)

test_segments = segment_df[
    segment_df["split"] == "test"
].reset_index(drop=True)

prediction_df = test_segments[
    ["segment_id", "original_id", "transcript"]
].copy()
prediction_df["prediction"] = predicted_text

FINAL_MODEL = DRIVE_ROOT / NEW_MODEL_NAME

if FINAL_MODEL.exists() and any(FINAL_MODEL.iterdir()) and not RESUME_TRAINING:
    raise RuntimeError(
        f"Refusing to overwrite an existing model directory on a fresh run: {FINAL_MODEL}. "
        "Rename/archive the existing v2 directory from the earlier run, then rerun this fresh v2 training."
    )

FINAL_MODEL.mkdir(parents=True, exist_ok=True)

print("Saving final model...", flush=True)
trainer.save_model(str(FINAL_MODEL))
processor.save_pretrained(str(FINAL_MODEL))

prediction_df.to_csv(
    FINAL_MODEL / "test_predictions.csv",
    index=False
)

segment_df.to_csv(
    FINAL_MODEL / "accepted_segments_manifest.csv",
    index=False
)

report_df.to_csv(
    FINAL_MODEL / "preprocessing_report.csv",
    index=False
)

pairs.to_csv(
    FINAL_MODEL / "source_pairs_and_splits.csv",
    index=False
)

summary = {
    "starting_model": str(EXISTING_MODEL),
    "new_model": str(FINAL_MODEL),
    "dataset_csv": str(DRIVE_CSV),
    "dataset_audio_folder": str(DRIVE_AUDIO),
    "selected_original_recordings": int(len(pairs)),
    "accepted_original_recordings": int(len(accepted_ids)),
    "failed_original_recordings": int(len(failed_ids)),
    "accepted_chunks": int(len(segment_df)),
    "train_chunks": int(len(processed["train"])),
    "validation_chunks": int(len(processed["validation"])),
    "test_chunks": int(len(processed["test"])),
    "num_train_epochs": int(NUM_EPOCHS),
    "train_batch_size": int(TRAIN_BATCH_SIZE),
    "gradient_accumulation_steps": int(GRADIENT_ACCUMULATION_STEPS),
    "evaluation_steps": int(EVAL_STEPS),
    "checkpoint_steps": int(SAVE_STEPS),
    "train_metrics": train_result.metrics,
    "validation_metrics": validation_results,
    "test_metrics": pred.metrics,
    "training_seconds": training_seconds,
    "validation_seconds": validation_seconds,
    "test_seconds": test_seconds,
    "preparation_seconds": PREPARATION_SECONDS,
    "final_model": str(FINAL_MODEL),
    "preprocessing_report": str(REPORT_FILE),
}

(FINAL_MODEL / "full_training_report.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False, default=float)
)

print("\n" + "=" * 70)
print("FINAL EXPORT COMPLETE")
print("=" * 70)
print("Final model:", FINAL_MODEL)
print("Validation:", validation_results)
print("Test:", pred.metrics)
print("Predictions:", FINAL_MODEL / "test_predictions.csv")

display(prediction_df)


## 15. What to inspect after incremental training

Important outputs:

- `incremental_recording_splits.csv`
- `preprocessing_sentence_safe_v2/full_segments_manifest.csv`
- `preprocessing_sentence_safe_v2/full_preprocessing_report.csv`
- `incremental_training/` checkpoints
- `final_whisper-small_medical_full_v2/test_predictions.csv`
- `final_whisper-small_medical_full_v2/full_training_report.json`

The original `final_whisper-small_medical_full` model remains untouched.


The incremental run is built on the existing trained model and the new conversations.

The notebook keeps the current sentence-safe preprocessing and automatic completeness gates. It requires an exact match between:

```text
CONV_001 ... CONV_NNN
```

in the CSV and:

```text
CONV_001.wav ... CONV_NNN.wav
```

in the audio folder.

The recording-level split is created before chunking, so chunks from the same conversation cannot leak across train, validation, and test.
